# Two Moons Bayesian-Torch — 5-Seed Evaluation

Same variants as `two_moons_bayesian.ipynb` but run over 5 random seeds. Checkpoints use `two_moons_bayesian_5seed/` and figures use the `5seed_two_moons_` prefix.

In [1]:
from pathlib import Path
import sys, os, time, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from bayesian_torch.models.dnn_to_bnn import dnn_to_bnn, get_kl_loss
from bayesian_torch.utils.avuc_loss import AvULoss

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "shared").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared import (TinyMLP, checkpoint_exists, load_checkpoint, save_checkpoint,
                    load_two_moons, seed_everything, train_map)
from shared.metrics import standard_metrics

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

SEEDS = [42, 711, 123, 456, 789]

FIGURES_DIR = os.path.join("..", "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

# Separate checkpoint dir — no collision with single-seed two_moons_bayesian/two_moons_{vname}_seed42.pt
CHECKPOINT_BASE = str(ROOT / "results" / "checkpoints" / "two_moons_bayesian_5seed")
os.makedirs(CHECKPOINT_BASE, exist_ok=True)

plt.rcParams.update({"font.family": "serif", "font.size": 12,
                     "savefig.dpi": 150, "savefig.bbox": "tight"})
print(f"5-seed Two Moons Bayesian | {len(SEEDS)} seeds | checkpoints -> {CHECKPOINT_BASE}")


Device: cuda
5-seed Two Moons Bayesian | 5 seeds | checkpoints -> /u/halle/carg/home_at/Documents/BNNs/results/checkpoints/two_moons_bayesian_5seed


In [2]:
def plot_boundary(predict_fn, X_data, y_data, title, ax=None):
    h = 0.05
    x_min, x_max = X_data[:,0].min()-0.5, X_data[:,0].max()+0.5
    y_min, y_max = X_data[:,1].min()-0.5, X_data[:,1].max()+0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    probs = predict_fn(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    if ax is None:
        _, ax = plt.subplots(figsize=(5,4))
    ax.contourf(xx, yy, probs, levels=50, cmap="RdBu_r", alpha=0.8, vmin=0, vmax=1)
    ax.contour(xx, yy, probs, levels=[0.5], colors="k", linewidths=1.5)
    ax.scatter(X_data[:,0], X_data[:,1], c=y_data, cmap="bwr",
               edgecolors="k", linewidths=0.4, s=30, zorder=3)
    ax.set_title(title)

def bnn_predict_samples(X_np, model, n_samples=100, batch_size=512):
    model.eval()
    parts = []
    with torch.no_grad():
        for i in range(0, len(X_np), batch_size):
            X_t = torch.tensor(X_np[i:i+batch_size], dtype=torch.float32).to(device)
            sims = [torch.softmax(model(X_t), dim=1).cpu().numpy() for _ in range(n_samples)]
            parts.append(np.stack(sims))
    return np.concatenate(parts, axis=1)

def bnn_predict(X_np, model, n_samples=30):
    return bnn_predict_samples(X_np, model, n_samples=n_samples).mean(axis=0)[:,1]

def build_bnn(model_type, moped=False, map_weights=None, moped_delta=0.5):
    net = TinyMLP(hidden=64).to(device)
    if moped and map_weights is not None:
        net.load_state_dict(map_weights)
    dnn_to_bnn(net, {
        "prior_mu": 0.0, "prior_sigma": 1.0,
        "posterior_mu_init": 0.0, "posterior_rho_init": -3.0,
        "type": model_type, "moped_enable": moped, "moped_delta": moped_delta,
    })
    return net.to(device)

def _val_nll(net, X_val, y_val, n_samples=10):
    val_probs = bnn_predict_samples(X_val, net, n_samples=n_samples).mean(axis=0)
    vp = np.clip(val_probs, 1e-12, 1.0)
    return float(-np.log(vp[np.arange(len(y_val)), y_val]).mean())

# 8 variants: Type x Init x Loss
variant_specs = []
for model_type, short in [("Flipout", "Flipout"), ("Reparameterization", "BBB")]:
    for moped in [False, True]:
        init = "moped" if moped else "scratch"
        for loss in ["elbo", "avuc"]:
            variant_specs.append({
                "variant_name": f"{short.lower()}_{init}_{loss}",
                "model_type": model_type, "moped": moped, "moped_delta": 0.5,
                "loss": loss, "epochs": 100 if moped else 300,
                "patience": 25, "lr": 5e-3, "avu_beta": 1.0,
                "plot_title": f"{short} / {init} / {'AvUC' if loss=='avuc' else 'ELBO'}",
            })
print(f"Configured {len(variant_specs)} BNN variants x {len(SEEDS)} seeds = "
      f"{len(variant_specs)*len(SEEDS)} total fits")


Configured 8 BNN variants x 5 seeds = 40 total fits


In [ ]:
all_seed_metrics = {}
all_seed_probs   = {}

criterion = nn.CrossEntropyLoss()

for seed in SEEDS:
    print(f"\n{'='*55}\nSEED {seed}  ({SEEDS.index(seed)+1}/{len(SEEDS)})\n{'='*55}")
    seed_everything(seed)

    data = load_two_moons(seed=seed, noise=0.3, batch_train=32, batch_eval=64, device=device)
    X_train = data["X_train"]; X_val = data["X_val"]; X_test = data["X_test"]
    y_train = data["y_train"]; y_val   = data["y_val"]; y_test = data["y_test"]
    train_loader = data["train_loader"]; val_loader = data["val_loader"]
    X_test_tensor = data["X_test_tensor"]
    train_ds = data["train_ds"]
    y_test_np = y_test
    num_samples = len(train_ds)

    # ----- MAP -----
    map_ckpt = os.path.join(CHECKPOINT_BASE, f"two_moons_map_seed{seed}.pt")
    map_model, _, _ = train_map(
        TinyMLP(hidden=64).to(device), train_loader,
        epochs=300, checkpoint_path=map_ckpt, device=device)
    map_model.eval()
    map_weights = copy.deepcopy(map_model.state_dict())

    seed_metrics, seed_probs = {}, {}
    for spec in variant_specs:
        vname = spec["variant_name"]
        ckpt = os.path.join(CHECKPOINT_BASE, f"two_moons_{vname}_seed{seed}.pt")
        net = build_bnn(spec["model_type"], spec["moped"],
                        map_weights=map_weights if spec["moped"] else None,
                        moped_delta=spec["moped_delta"])

        if checkpoint_exists(ckpt):
            payload = load_checkpoint(ckpt, map_location="cpu")
            if isinstance(payload, dict) and "model_state_dict" in payload:
                net.load_state_dict(payload["model_state_dict"])
                v_metrics = dict(payload.get("metrics", {}))
                print(f"  {vname}: loaded")
        else:
            use_avuc = spec["loss"] == "avuc"
            avu_loss_fn = AvULoss(beta=spec["avu_beta"]) if use_avuc else None
            n_mc = 5
            opt = optim.Adam(net.parameters(), lr=spec["lr"])
            best_val, best_state, wait, ran = float("inf"), None, 0, 0
            t0 = time.time()
            for epoch in range(spec["epochs"]):
                net.train()
                for inputs, labels in train_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    opt.zero_grad()
                    if use_avuc:
                        logits_mc = torch.stack([net(inputs) for _ in range(n_mc)], dim=0)
                        mean_logits = logits_mc.mean(dim=0)
                        ce = criterion(mean_logits, labels)
                        kl = get_kl_loss(net) / num_samples
                        probs = torch.softmax(mean_logits, dim=1)
                        ent = -(probs * torch.log(probs + 1e-12)).sum(dim=1)
                        avu = avu_loss_fn(mean_logits, labels, ent.mean().item(), type=0).squeeze()
                        loss = ce + kl + avu
                    else:
                        loss = criterion(net(inputs), labels) + get_kl_loss(net) / num_samples
                    loss.backward(); opt.step()
                ran = epoch + 1
                val_nll = _val_nll(net, X_val, y_val)
                if val_nll < best_val - 1e-4:
                    best_val, best_state, wait = val_nll, copy.deepcopy(net.state_dict()), 0
                else:
                    wait += 1
                    if wait >= spec["patience"]: break
            if best_state is not None:
                net.load_state_dict(best_state)
            fit_time = time.time()-t0
            v_metrics = {
                "Model_Type": spec["model_type"],
                "Init": "MOPED" if spec["moped"] else "scratch",
                "Loss": "ELBO+AvUC" if use_avuc else "ELBO",
                "Epochs": ran, "Best_Val_NLL": best_val,
                "LR": spec["lr"], "Fit_Time (s)": fit_time,
            }
            save_checkpoint({"model_state_dict": net.state_dict(), "metrics": v_metrics}, ckpt)
            print(f"  {vname}: trained ({fit_time:.1f}s, {ran} epochs)")

        samples = bnn_predict_samples(X_test, net, n_samples=50)
        v_test_probs = samples.mean(axis=0)
        row = {**v_metrics, **standard_metrics(v_test_probs, y_test_np)}
        seed_metrics[vname] = row
        seed_probs[vname]   = v_test_probs
        print(f"    Acc={row.get('Accuracy',float('nan')):.4f}  "
              f"NLL={row.get('NLL',float('nan')):.4f}  "
              f"ECE={row.get('ECE',float('nan')):.4f}")

    all_seed_metrics[seed] = seed_metrics
    all_seed_probs[seed]   = seed_probs

print(f"\nCompleted {len(SEEDS)} seeds x {len(variant_specs)} variants.")



SEED 42  (1/5)
  flipout_scratch_elbo: trained (16.2s, 30 epochs)
    Acc=0.9220  NLL=0.1926  ECE=0.0139
  flipout_scratch_avuc: trained (77.5s, 34 epochs)
    Acc=0.8680  NLL=0.3027  ECE=0.0203
  flipout_moped_elbo: trained (24.1s, 47 epochs)
    Acc=0.9210  NLL=0.1954  ECE=0.0083
  flipout_moped_avuc: trained (159.0s, 46 epochs)
    Acc=0.9240  NLL=0.1936  ECE=0.0114
  bbb_scratch_elbo: trained (25.7s, 39 epochs)
    Acc=0.9230  NLL=0.1974  ECE=0.0146
  bbb_scratch_avuc: trained (172.5s, 48 epochs)
    Acc=0.9210  NLL=0.2007  ECE=0.0118
  bbb_moped_elbo: trained (24.9s, 36 epochs)
    Acc=0.9220  NLL=0.1942  ECE=0.0132
  bbb_moped_avuc: trained (93.0s, 26 epochs)
    Acc=0.9245  NLL=0.1965  ECE=0.0127

SEED 711  (2/5)
  flipout_scratch_elbo: trained (42.5s, 64 epochs)
    Acc=0.8675  NLL=0.3110  ECE=0.0272
  flipout_scratch_avuc: trained (193.0s, 47 epochs)
    Acc=0.9185  NLL=0.2018  ECE=0.0161
  flipout_moped_elbo: trained (28.0s, 43 epochs)
    Acc=0.9240  NLL=0.2011  ECE=0.0133


In [ ]:
metric_keys = ["Accuracy", "NLL", "Brier_Score", "ECE"]

agg = {}
for spec in variant_specs:
    vname = spec["variant_name"]; agg[vname] = {}
    for mk in metric_keys:
        vals = [all_seed_metrics[s][vname].get(mk, float("nan")) for s in SEEDS]
        vals = [v for v in vals if not np.isnan(v)]
        agg[vname][f"{mk}_mean"] = float(np.mean(vals)) if vals else float("nan")
        agg[vname][f"{mk}_std"]  = float(np.std(vals))  if len(vals)>1 else 0.0

rows = []
for spec in variant_specs:
    vname = spec["variant_name"]
    row = {mk: f"{agg[vname][f'{mk}_mean']:.4f} ± {agg[vname][f'{mk}_std']:.4f}"
           for mk in metric_keys}
    rows.append(row)
summary_df = pd.DataFrame(rows, index=[s["plot_title"] for s in variant_specs])
summary_df.index.name = "Variant"
print(f"Two Moons Bayesian — {len(SEEDS)}-seed metrics (mean ± std)")
display(summary_df)

agg_flat = [{"Variant": s["plot_title"], **agg[s["variant_name"]]} for s in variant_specs]
agg_df = pd.DataFrame(agg_flat).set_index("Variant")
os.makedirs("results/metrics", exist_ok=True)
agg_df.to_csv("results/metrics/two_moons_bayesian_5seed_metrics.csv")
print("\nSaved to results/metrics/two_moons_bayesian_5seed_metrics.csv")


In [ ]:
plot_metrics = ["Accuracy", "NLL", "ECE"]
titles = [s["plot_title"] for s in variant_specs]
x = np.arange(len(titles))

for mk in plot_metrics:
    means = [agg[s["variant_name"]][f"{mk}_mean"] for s in variant_specs]
    stds  = [agg[s["variant_name"]][f"{mk}_std"]  for s in variant_specs]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x, means, yerr=stds, capsize=4, color="steelblue", alpha=0.7, edgecolor="k")
    ax.set_xticks(x)
    ax.set_xticklabels([t.replace(" / ", "\n") for t in titles], fontsize=7)
    ax.set_ylabel(mk)
    ax.set_title(f"Two Moons Bayesian — {mk}  (mean ± std, {len(SEEDS)} seeds)")
    ax.grid(True, alpha=0.3, axis="y")
    fname = f"5seed_two_moons_bayesian_{mk.lower()}_bar.png"
    fig.savefig(os.path.join(FIGURES_DIR, fname), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {fname}")


In [ ]:
# Per-variant decision boundary: representative (last seed)
for spec in variant_specs:
    vname = spec["variant_name"]; title = spec["plot_title"]

    ckpt = os.path.join(CHECKPOINT_BASE, f"two_moons_{vname}_seed{SEEDS[-1]}.pt")
    payload = load_checkpoint(ckpt, map_location="cpu")
    if not (isinstance(payload, dict) and "model_state_dict" in payload):
        print(f"  Skipping {vname}"); continue

    net = build_bnn(spec["model_type"], spec["moped"])
    net.load_state_dict(payload["model_state_dict"])

    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    plot_boundary(lambda X: bnn_predict(X, net), X_train, y_train,
                  f"{title} — 5-Seed Rep. Decision Boundary", ax=ax)
    fig.savefig(os.path.join(FIGURES_DIR, f"5seed_two_moons_{vname}_boundary.png"),
                dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: 5seed_two_moons_{vname}_boundary.png")
print(f"\nDone: {len(variant_specs)} boundary figures")


In [ ]:
# Per-variant OOD epistemic maps: mean across seeds
ext = 4.5
gx, gy = np.meshgrid(np.arange(-ext, ext, 0.1), np.arange(-ext, ext, 0.1))
ood_grid_np = np.c_[gx.ravel(), gy.ravel()]

def _entropy(p):
    return -(p * np.log(p + 1e-12)).sum(axis=-1)

def epistemic_bnn(model, X_np, n_samples=50, batch_size=512):
    samples = bnn_predict_samples(X_np, model, n_samples=n_samples, batch_size=batch_size)
    mean_p = samples.mean(axis=0)
    return _entropy(mean_p) - _entropy(samples).mean(axis=0)

for spec in variant_specs:
    vname = spec["variant_name"]; title = spec["plot_title"]

    all_epi_grids = []
    for seed in SEEDS:
        ckpt = os.path.join(CHECKPOINT_BASE, f"two_moons_{vname}_seed{seed}.pt")
        payload = load_checkpoint(ckpt, map_location="cpu")
        if isinstance(payload, dict) and "model_state_dict" in payload:
            net = build_bnn(spec["model_type"], spec["moped"])
            net.load_state_dict(payload["model_state_dict"])
            epi = epistemic_bnn(net, ood_grid_np, n_samples=30)
            all_epi_grids.append(epi)
    if not all_epi_grids:
        continue
    mean_epi = np.stack(all_epi_grids).mean(axis=0).reshape(gx.shape)

    fig, ax = plt.subplots(1, 1, figsize=(8, 7))
    cf = ax.contourf(gx, gy, mean_epi, levels=50, cmap="YlOrRd")
    ax.scatter(X_train[:,0], X_train[:,1], c=y_train, cmap="bwr",
               s=8, alpha=0.25, edgecolors="none")
    plt.colorbar(cf, ax=ax, label="Epistemic uncertainty (MI)")
    ax.set_title(f"{title} — 5-Seed Mean OOD Epistemic Uncertainty")
    ax.set_xlabel("x1"); ax.set_ylabel("x2")
    fig.savefig(os.path.join(FIGURES_DIR, f"5seed_two_moons_{vname}_ood_epistemic.png"),
                dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: 5seed_two_moons_{vname}_ood_epistemic.png")
print(f"\nDone: {len(variant_specs)} OOD epistemic figures")


In [ ]:
# Per-variant reliability diagrams — mean ± std across seeds (ECE-style: confidence vs accuracy)
n_bins = 15
bins = np.linspace(0, 1, n_bins + 1)

for spec in variant_specs:
    vname = spec["variant_name"]
    title = spec["plot_title"]

    all_accs, all_confs = [], []
    for seed in SEEDS:
        probs = all_seed_probs.get(seed, {}).get(vname)
        if probs is None:
            continue
        data_s = load_two_moons(seed=seed, noise=0.3, batch_train=32, batch_eval=64, device=device)
        y_true = data_s["y_test"]
        conf = probs.max(axis=1)
        pred = probs.argmax(axis=1)
        accs_bin, confs_bin = [], []
        for lo, hi in zip(bins[:-1], bins[1:]):
            mask = (conf > lo) & (conf <= hi)
            if mask.sum() == 0:
                accs_bin.append(np.nan)
                confs_bin.append((lo + hi) / 2)
            else:
                accs_bin.append(float((pred[mask] == y_true[mask]).mean()))
                confs_bin.append(float(conf[mask].mean()))
        all_accs.append(accs_bin)
        all_confs.append(confs_bin)

    all_accs  = np.array(all_accs)
    all_confs = np.array(all_confs)
    mean_obs  = np.nanmean(all_accs, axis=0)
    std_obs   = np.nanstd(all_accs, axis=0)
    mean_conf = np.nanmean(all_confs, axis=0)
    valid = ~np.isnan(mean_obs)

    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
    ax.plot(mean_conf[valid], mean_obs[valid], "o-", label=f"Model (mean, {len(SEEDS)} seeds)")
    ax.fill_between(mean_conf[valid],
                    (mean_obs - std_obs)[valid],
                    (mean_obs + std_obs)[valid],
                    alpha=0.2, label=f"±1σ across {len(SEEDS)} seeds")
    ax.set_xlabel("Mean Confidence")
    ax.set_ylabel("Accuracy")
    ax.set_title(f"{title} — Reliability Diagram ({len(SEEDS)} seeds)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    fig.savefig(os.path.join(FIGURES_DIR, f"5seed_two_moons_{vname}_reliability.png"),
                dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: 5seed_two_moons_{vname}_reliability.png")
print(f"\nDone: {len(variant_specs)} reliability figures")